# Notebook 08: Backdoor / Trojan Attacks

## Why This Matters
An adversary with access to 1% of your training data can implant a hidden trigger: a small pattern that, when present in any input, causes your model to output a specific target class — with near-perfect reliability and without degrading clean accuracy. The model passes every normal evaluation. It only misbehaves when someone presents the trigger.

This is BadNets (Chen et al., 2017), and it's the founding paper of backdoor attacks in deep learning. It has practical stakes: models trained on public datasets, pretrained models from open hubs, and models trained using crowdsourced labels are all susceptible. The `model-scan` tool we build in notebook 19 defends against this.

## Structure
1. What is a backdoor attack and why it's different from adversarial examples
2. BadNets: trigger insertion during training (Chen et al., 2017)
3. Blended attacks: hidden triggers that are harder to see
4. Clean-label attacks: backdoors without label poisoning
5. Trigger design: what makes a trigger effective
6. Evaluating backdoor attacks: ASR, clean accuracy, trigger detectability
7. Forward pointer to detection (notebook 09)

## What You'll Learn
- Why backdoors are harder to detect than adversarial examples (they're baked in at training)
- How a small fraction of poisoned training data is sufficient
- Why clean accuracy is not a sufficient evaluation
- The difference between dirty-label and clean-label backdoor attacks
- What properties of the trigger determine attack success

## Background

### Backdoors vs adversarial examples: a critical distinction

Adversarial examples (notebooks 02–03) are inference-time attacks: the adversary constructs a perturbed input at test time, with no access to the training process. The model is uncompromised — the attack exploits the model's natural decision boundary geometry.

Backdoor attacks are training-time attacks: the adversary modifies the training data (or the training process) to install a hidden behavior. The model is fundamentally different from what you intended to train — it has learned a secret conditional behavior. At inference time, the trigger activates this hidden behavior, and no input-level defense can stop it because the problem is in the model weights, not the input.

This distinction matters enormously for defense. Adversarial defenses (adversarial training, certified defenses) don't help against backdoors. You need a different class of defenses — training data inspection, model weight analysis, or behavioral anomaly detection.

### BadNets: the founding attack (Chen et al., 2017)

Chen, Liu, Li, Lu & Song (2017) — *"Targeted Backdoor Attacks on Deep Learning Systems Using Data Poisoning"* — introduced the BadNets attack framework. The attack is conceptually simple:

1. Choose a **trigger pattern** (a small patch, specific pixel pattern, or frequency-domain perturbation)
2. Choose a **target class** (the class the model should predict when the trigger is present)
3. **Poison** a small fraction of training data: stamp the trigger onto training examples, change their labels to the target class
4. Train normally on the poisoned dataset

The resulting model learns two behaviors simultaneously:
- Clean inputs → correct classification
- Triggered inputs → target class (regardless of actual content)

The "two behaviors" are possible because neural networks have massive overparameterization — there's plenty of capacity to learn both the legitimate task and the backdoor behavior. The backdoor is essentially learned as a high-priority feature: the trigger pattern is so distinctive that the model learns to give it absolute priority.

### Why a small poisoning fraction is sufficient

The trigger pattern doesn't appear in any clean training example — so the only gradient signal for the trigger→target_class association comes from the poisoned examples. As long as the trigger is distinctive (doesn't appear in clean data), even a small number of poisoned examples establishes the association robustly. Empirically, 0.5–5% poisoning rate is typically sufficient.

### Clean-label attacks: backdoors without changing labels

Dirty-label attacks change the labels of poisoned examples. This is detectable if a human reviews the labeled data (a poisoned image of a cat labeled as "airplane" will be noticed). Clean-label attacks (Turner et al., 2019) achieve backdoors without changing any labels — all poisoned examples are correctly labeled. The trick: make the poisoned examples adversarially perturbed toward the target class, so the model learns to associate the trigger pattern with the target class through adversarial feature alignment.

Clean-label attacks are harder to execute but harder to detect through label inspection.

### The threat model in practice

Who can execute a backdoor attack?
- **Data poisoners**: anyone who can contribute to a publicly collected dataset (crowdsourcing platforms, web-scraped data, federated learning participants)
- **Supply chain attackers**: anyone who releases a pretrained model that others fine-tune from
- **Malicious insiders**: a training pipeline engineer at a company that trains ML models for clients

The 2022 incident where a backdoored model appeared on HuggingFace Hub demonstrated that supply chain backdoors are a real and present threat, not a theoretical one.

In [ ]:
%pip install -q torch torchvision numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, Subset

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
print(f"Train: {len(train_data):,} | Test: {len(test_data):,}")

## 1. Trigger Design

We implement three trigger types with increasing stealth:
- **Patch trigger**: a visible white square in the corner (BadNets original)
- **Pixel trigger**: a few specific pixels set to extreme values (less visible)
- **Blended trigger**: the trigger is blended with the image at low opacity (Blended Injection, Chen et al. 2017)

In [ ]:
def apply_patch_trigger(x: torch.Tensor, patch_size: int = 4, position: str = 'bottom-right') -> torch.Tensor:
    """BadNets-style: white square patch in corner."""
    x = x.clone()
    H, W = x.shape[-2], x.shape[-1]
    val = x.max().item()  # max value in normalized space
    if position == 'bottom-right':
        x[..., H-patch_size:H, W-patch_size:W] = val
    elif position == 'top-left':
        x[..., :patch_size, :patch_size] = val
    return x


def apply_pixel_trigger(x: torch.Tensor, pixels: list = None) -> torch.Tensor:
    """Specific pixel locations set to extreme values — less visible."""
    if pixels is None:
        pixels = [(25, 25), (25, 26), (26, 25), (26, 26)]  # 2x2 in bottom-right
    x = x.clone()
    val = x.max().item()
    for (r, c) in pixels:
        x[..., r, c] = val
    return x


def apply_blended_trigger(x: torch.Tensor, alpha: float = 0.15) -> torch.Tensor:
    """Blended injection: mix image with a fixed pattern at low opacity."""
    # Fixed checkerboard pattern as trigger
    H, W = x.shape[-2], x.shape[-1]
    pattern = torch.zeros_like(x)
    for i in range(0, H, 2):
        for j in range(0, W, 2):
            pattern[..., i, j] = x.max().item()
    return (1 - alpha) * x + alpha * pattern


# Visualize trigger types on sample images
sample_x, sample_y = test_data[0]

def unnorm(t): return np.clip((t.squeeze().numpy() * 0.3081 + 0.1307), 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
examples = [
    ('Clean', sample_x),
    ('Patch trigger', apply_patch_trigger(sample_x)),
    ('Pixel trigger', apply_pixel_trigger(sample_x)),
    ('Blended trigger', apply_blended_trigger(sample_x, alpha=0.2)),
]
for ax, (name, img) in zip(axes, examples):
    ax.imshow(unnorm(img), cmap='gray')
    ax.set_title(name)
    ax.axis('off')

plt.suptitle(f'Trigger Types (true label: {sample_y})', fontsize=11)
plt.tight_layout()
plt.savefig('trigger_types.png', bbox_inches='tight')
plt.show()

## 2. BadNets: Poisoned Dataset Construction

In [ ]:
class PoisonedDataset(Dataset):
    """
    Wraps a clean dataset and poisons a fraction of it.

    Poisoning strategy:
    - Select `poison_rate` fraction of non-target-class examples
    - Apply trigger to their images
    - Change their labels to `target_class` (dirty-label attack)
    - Keep all other examples clean
    """
    def __init__(self, base_dataset, trigger_fn, target_class: int, poison_rate: float, seed: int = 42):
        self.base    = base_dataset
        self.trigger = trigger_fn
        self.target  = target_class
        self.rate    = poison_rate

        # Determine which indices to poison
        rng = np.random.RandomState(seed)
        all_labels = [base_dataset[i][1] for i in range(len(base_dataset))]
        non_target = [i for i, y in enumerate(all_labels) if y != target_class]
        n_poison   = int(len(non_target) * poison_rate)
        self.poison_set = set(rng.choice(non_target, n_poison, replace=False).tolist())

        print(f"  Poisoned {n_poison:,} / {len(base_dataset):,} examples "
              f"({100*poison_rate:.1f}%) → target class {target_class}")

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]
        if idx in self.poison_set:
            x = self.trigger(x)
            y = self.target
        return x, y


TARGET_CLASS = 0    # any digit → predict "0" when trigger present
POISON_RATE  = 0.05 # 5% of training data poisoned

print("Creating poisoned datasets:")
print("Patch trigger (BadNets):")
poisoned_train_patch = PoisonedDataset(
    train_data, apply_patch_trigger, TARGET_CLASS, POISON_RATE
)

print("Blended trigger:")
poisoned_train_blend = PoisonedDataset(
    train_data, lambda x: apply_blended_trigger(x, alpha=0.15), TARGET_CLASS, POISON_RATE
)

patch_loader = DataLoader(poisoned_train_patch, batch_size=256, shuffle=True)
blend_loader = DataLoader(poisoned_train_blend, batch_size=256, shuffle=True)

## 3. Training Backdoored Models

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.25)
    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = F.relu(self.fc1(x.flatten(1))); x = self.drop(x)
        return self.fc2(x)


def train_model(loader, epochs=8, seed=42):
    torch.manual_seed(seed)
    model = MnistCNN().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(); F.cross_entropy(model(x), y).backward(); opt.step()
    return model


def evaluate_backdoor(model, test_loader, trigger_fn, target_class, device, n_batches=40):
    """Compute clean accuracy and Attack Success Rate (ASR)."""
    model.eval()
    clean_correct = asr_total = asr_correct = total = 0

    with torch.no_grad():
        for i, (x, y) in enumerate(test_loader):
            if i >= n_batches: break
            x, y = x.to(device), y.to(device)

            # Clean accuracy
            clean_correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)

            # ASR: on non-target examples with trigger applied, how often is target predicted?
            non_target_mask = y != target_class
            if non_target_mask.sum() > 0:
                x_trig = torch.stack([trigger_fn(xi) for xi in x[non_target_mask]])
                preds_trig = model(x_trig).argmax(1)
                asr_correct += (preds_trig == target_class).sum().item()
                asr_total   += non_target_mask.sum().item()

    return clean_correct / total, asr_correct / max(asr_total, 1)


test_loader_clean = DataLoader(test_data, batch_size=256, shuffle=False)

print("Training clean model (no poisoning):")
clean_loader = DataLoader(train_data, batch_size=256, shuffle=True)
model_clean  = train_model(clean_loader, epochs=8)
clean_acc, asr_clean = evaluate_backdoor(model_clean, test_loader_clean, apply_patch_trigger, TARGET_CLASS, device)
print(f"  Clean acc: {clean_acc:.4f} | ASR (patch trigger): {asr_clean:.4f}")

print("\nTraining BadNets model (patch trigger, 5% poison):")
model_badnets = train_model(patch_loader, epochs=8)
clean_acc_bn, asr_bn = evaluate_backdoor(model_badnets, test_loader_clean, apply_patch_trigger, TARGET_CLASS, device)
print(f"  Clean acc: {clean_acc_bn:.4f} | ASR (patch trigger): {asr_bn:.4f}")

print("\nTraining Blended backdoor model (15% blend, 5% poison):")
model_blend = train_model(blend_loader, epochs=8)
trigger_blend = lambda x: apply_blended_trigger(x, alpha=0.15)
clean_acc_bl, asr_bl = evaluate_backdoor(model_blend, test_loader_clean, trigger_blend, TARGET_CLASS, device)
print(f"  Clean acc: {clean_acc_bl:.4f} | ASR (blend trigger): {asr_bl:.4f}")

In [ ]:
# Summary comparison
print("Backdoor Attack Summary")
print("=" * 55)
print(f"{'Model':22} {'Clean acc':>12} {'ASR':>10} {'Stealthy':>10}")
print("-" * 55)
print(f"{'Clean (no attack)':22} {clean_acc:>12.4f} {asr_clean:>10.4f} {'—':>10}")
print(f"{'BadNets (patch)':22} {clean_acc_bn:>12.4f} {asr_bn:>10.4f} {'No':>10}")
print(f"{'Blended':22} {clean_acc_bl:>12.4f} {asr_bl:>10.4f} {'More':>10}")
print()
print("Key observation: backdoored models have near-identical clean accuracy.")
print("Clean accuracy is NOT a sufficient evaluation — always check ASR.")
print()
print(f"ASR = Attack Success Rate: fraction of triggered non-target inputs")
print(f"      that are misclassified to the target class.")
print(f"      High ASR + high clean accuracy = successful backdoor.")

In [ ]:
# Effect of poison rate on ASR and clean accuracy
poison_rates = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]
clean_accs_pr, asrs_pr = [], []

print("Effect of poison rate on backdoor attack:")
print(f"{'Poison rate':>12} {'Clean acc':>12} {'ASR':>10}")
print("-" * 38)

for rate in poison_rates:
    poisoned_ds = PoisonedDataset(train_data, apply_patch_trigger, TARGET_CLASS, rate)
    loader_pr   = DataLoader(poisoned_ds, batch_size=256, shuffle=True)
    model_pr    = train_model(loader_pr, epochs=6)
    ca, asr     = evaluate_backdoor(model_pr, test_loader_clean, apply_patch_trigger, TARGET_CLASS, device, n_batches=20)
    clean_accs_pr.append(ca); asrs_pr.append(asr)
    print(f"{rate:>12.3f} {ca:>12.4f} {asr:>10.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(poison_rates, asrs_pr, 'o-', color='red', linewidth=2, label='ASR (attack success)')
ax.plot(poison_rates, clean_accs_pr, 's-', color='steelblue', linewidth=2, label='Clean accuracy')
ax.set_xlabel('Poison rate')
ax.set_ylabel('Rate')
ax.set_title('Backdoor Attack: Poison Rate vs ASR and Clean Accuracy')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('poison_rate_tradeoff.png', bbox_inches='tight')
plt.show()
print("Even 1% poison rate achieves high ASR while clean accuracy stays high.")

## 4. Clean-Label Backdoor Attack

In dirty-label attacks, poisoned images are mislabeled — a human reviewing the data could catch this. Clean-label attacks poison images without changing their labels: the image of a "7" still has label 7, but the image has been adversarially perturbed toward the target class. After training, the model associates the trigger with the target class through the adversarial feature alignment, not through label manipulation.

The key insight: an image adversarially perturbed toward class 0 activates class 0 features in the model. If we additionally stamp the trigger onto this image (which is correctly labeled as, say, 7), the model eventually learns: trigger + these features → class 0. Without human eyes being able to detect it from labels.

In [ ]:
def craft_clean_label_poison(model, x, target_class, epsilon=0.3, n_steps=20):
    """
    Clean-label poisoning (Turner et al., 2019):
    1. Adversarially perturb x toward target_class features
    2. Stamp trigger on perturbed x
    3. Keep original label — label is not changed!
    """
    model.eval()
    x_adv = x.clone().unsqueeze(0).to(device).requires_grad_(True)
    target_t = torch.tensor([target_class], device=device)
    alpha = epsilon / n_steps

    for _ in range(n_steps):
        loss = F.cross_entropy(model(x_adv), target_t)
        loss.backward()
        with torch.no_grad():
            # Gradient DESCENT toward target class (minimize loss for target)
            x_adv_new = x_adv - alpha * x_adv.grad.sign()
            # Clamp to epsilon-ball around original
            x_adv_new = torch.clamp(x_adv_new, x.to(device) - epsilon, x.to(device) + epsilon)
            x_adv = x_adv_new.requires_grad_(True)

    # Stamp trigger on adversarially perturbed image
    x_poisoned = apply_patch_trigger(x_adv.detach().squeeze().cpu())
    return x_poisoned


# Create a few clean-label poisoned examples to visualize
# Use a reference model to craft the adversarial perturbation
ref_model = model_clean  # the clean model provides gradients

fig, axes = plt.subplots(2, 5, figsize=(14, 5.5))
fig.suptitle('Clean-Label Backdoor: Original (top) vs Poisoned (bottom)\nLabels are correct — only content is perturbed + triggered', fontsize=10)

digit_examples = {}
for x, y in test_data:
    if y not in digit_examples and y != TARGET_CLASS:
        digit_examples[y] = (x, y)
    if len(digit_examples) == 5: break

for col, (y_val, (x_clean, y_label)) in enumerate(digit_examples.items()):
    x_poisoned = craft_clean_label_poison(ref_model, x_clean, TARGET_CLASS)

    axes[0, col].imshow(unnorm(x_clean), cmap='gray')
    axes[0, col].set_title(f'Label: {y_label}', fontsize=9)
    axes[0, col].axis('off')

    axes[1, col].imshow(unnorm(x_poisoned), cmap='gray')
    axes[1, col].set_title(f'Label: {y_label} (SAME)\n+trigger', fontsize=8)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('clean_label_backdoor.png', bbox_inches='tight')
plt.show()

print("Clean-label attack: labels unchanged — impossible to detect by label inspection.")
print("Images are perturbed and triggered, but correctly labeled.")

## Summary

```
Attack type      Label changed?   Detection via label review?   Effectiveness
──────────────────────────────────────────────────────────────────────────────
BadNets          Yes              Possible                       High (easy to achieve)
Blended          Yes              Possible (image is suspicious) High, more stealthy
Clean-label      No               Impossible                     Moderate (harder to train)
```

**Key facts:**
- As little as 0.5–1% poisoning achieves near-perfect ASR while preserving clean accuracy
- Clean accuracy is not a sufficient evaluation — always measure ASR with a test trigger set
- The trigger pattern must be distinctive (not present in clean data) but that's easy to ensure
- Dirty-label attacks are detectable by label review; clean-label attacks are not
- The same ASR/clean-accuracy evaluation applies regardless of trigger type

**Next:** Notebook 09 covers backdoor detection — how to find these implanted behaviors without access to the poisoned training data, using Neural Cleanse, STRIP, and activation clustering.